Guardrails AI

Guardrails AI is a framework that checks, validates, and controls AI inputs and outputs before they reach the user.

User
   │
   ▼
LLM 
   │
   ▼
Guardrails
   │
   ▼
Safe Response

It acts like a quality inspector

Input Guardrails

Before AI answers.

Checks

Prompt injection,
Bad language,
Sensitive information,
SQL injection,
Unsafe requests.

Output Guardrails

After AI answers.

Checks

Correct JSON,
Hallucinations (where possible),
Toxic content,
PII,
Response length.

Validators are simply rules.

Common Validators
1. JSON Validator

Checks whether output is valid JSON.

Good

{
"name":"John",
"age":25
}

Bad

name:John
age:25

Regex Validator

Checks a pattern.

Phone number

9876543210

Valid

98765

Invalid.

PII Validator

Detects

Aadhaar
PAN
Credit card
Phone number
Email
Passport

Toxicity Validator

Checks for

abusive language
hate speech
offensive words

Rail Spec = Configuration that defines what valid AI input/output looks like.

Rail Specs tell Guardrails

Output must be JSON,
Name is required,
Age is integer,
Email must be valid.

In [1]:
from guardrails import Guard

# ReAct- Reason + Action
Thinks about what to do
Uses a tool if needed
Observes the result
Thinks again
Continues until it has enough information
=>
Question
   ↓
Reason
   ↓
Act (Tool Call)
   ↓
Observe Result
   ↓
Reason Again
   ↓
Act Again
   ↓
Observe
   ↓
Final Answer

# failure of agents
Failure 1: Tool Not Available
        using tool that is not available in the code.
Failure 2: Wrong Tool Selection
Failure 3; Infinte loops
         Reason
         Search
         Reason 
         Search 
         Reason
Failure 4: Graceful Failure
        instead crashing into an exception directly , goes to try and catch and returns a proper failure
Failure 5; Retry Logic

In [2]:
import sqlite3
from duckduckgo_search import DDGS

# create the data base for DB tool 

In [3]:
con = sqlite3.connect("employee.db")
 
cursor=con.cursor()

cursor.execute(""" CREATE TABLE IF NOT EXISTS employee( 
              employee_id INTEGER,
              employee_name TEXT,
              manager TEXT
              )
            """)
cursor.execute("DELETE FROM employee")
cursor.execute(""" INSERT INTO employee VALUES (01, 'vinnu' ,'hari')
               """)

cursor.execute(""" INSERT INTO employee VALUES (02, 'yuri' ,'aries')
               """)

con.commit()
con.close()
print("database ready")

database ready


# database tool

In [4]:
def db_query(employee_id):
    con = sqlite3.connect("employee.db")
    cursor= con.cursor()
    cursor.execute("""
                SELECT employee_name, manager 
                FROM employee WHERE employee_id=? """, (int(employee_id),)
                )
    result =  cursor.fetchone()

    con.close()

    if result:
        return  (
            f"Employee:{result[0]},"
            f"Manager:{result[1]}"
        )
    return "Employee not found"

In [5]:
db_query(101)

'Employee not found'

In [6]:
from ddgs import DDGS

In [6]:
#from duckduckgo_search import DDGS
from ddgs import DDGS


def web_search(query):
    """Perform a DuckDuckGo search and return the top result as a readable string."""
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=5))

    if not results:
        print("No web results found.")
        return "No result found"

    formatted = []
    for item in results:
        formatted.append({
            "title": item.get("title", "No title"),
            "snippet": item.get("body", item.get("snippet", "")),
            "url": item.get("href", item.get("url", item.get("link", "No URL")))
        })

    print("Web search results:")
    for i, item in enumerate(formatted, 1):
        print(f"{i}. {item['title']}")
        print(f"   {item['url']}")
        print(f"   {item['snippet'][:120]}...")
        print()

    top = formatted[0]
    return (
        f"Top result: {top['title']}\n"
        f"URL: {top['url']}\n"
        f"Snippet: {top['snippet'][:250]}..."
    )


def pdf_lookup(query, pdf_path="project_report_full.pdf"):
    """Search project_report_full.pdf for the requested query and return a snippet."""
    import os
    from pypdf import PdfReader

    if not os.path.exists(pdf_path):
        return f"PDF not found: {pdf_path}"

    reader = PdfReader(pdf_path)

    document_text = []
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            document_text.append(page_text)

    full_text = "\n".join(document_text)

    query_lower = query.lower()
    index = full_text.lower().find(query_lower)

    if index == -1:
        return f"No match found in {pdf_path} for '{query}'"

    start = max(0, index - 200)
    end = min(len(full_text), index + 500)

    snippet = full_text[start:end].replace("\n", " ")

    return f"Found in {pdf_path}: ...{snippet}..."

In [7]:
web_search("who won ipl 2026?")

Web search results:
1. 2026 Indian Premier League - Wikipedia
   https://en.wikipedia.org/wiki/2026_Indian_Premier_League
   2026 Indian Premier League ... The 2026 Indian Premier League, also known as IPL 19 and branded as TATA IPL 2026, was th...

2. Who won IPL 2026? Full list of Indian Premier League winners, runners ...
   https://www.sportingnews.com/in/cricket/news/who-won-ipl-2026-list-indian-premier-league-winners-runners/c72cd80b310493e99b2306e0
   Full list of Indian Premier League winners, runners-up Chennai Super Kings and Mumbai Indians are the most successful te...

3. Who Won What in IPL 2026: Full List of Winners and Prize Money - Sports ...
   https://www.financialexpress.com/sports/ipl-2026-final-full-list-winners-prize-money-vaibhav-sooryavanshi-bank-breaking-show/4255808/
   Who won what in IPL 2026: Vaibhav Sooryavanshi steals the show even as RCB pocket Rs 20 cr While the champions take home...

4. IPL 2026 final: Who won the orange and purple caps? Full list of 

'Top result: 2026 Indian Premier League - Wikipedia\nURL: https://en.wikipedia.org/wiki/2026_Indian_Premier_League\nSnippet: 2026 Indian Premier League ... The 2026 Indian Premier League, also known as IPL 19 and branded as TATA IPL 2026, was the 19th edition of the Indian Premier League, a professional Twenty20 cricket league. The tournament featured 10 teams competing in...'

In [6]:
web_search("linkedin profile of microsoft ceo?")

Web search results:
1. Satya Nadella - Wikipedia
   https://en.wikipedia.org/wiki/Satya_Nadella
   Satya Narayana Nadella (born 19 August 1967) is an Indian-American business executive. He is the chairman and chief exec...

2. Satya Nadella - Chairman and CEO at Microsoft - LinkedIn
   https://www.linkedin.com/in/satyanadella
   As chairman and CEO of Microsoft, I define my mission and that of my company as empowering every person and every organi...

3. Ryan Roslansky - Executive Vice President at Microsoft - LinkedIn
   https://www.linkedin.com/in/ryanroslansky
   Ryan Roslansky is responsible for LinkedIn, Microsoft Office, Teams and Outlook and a member of Satya Nadella's senior l...

4. Satya Nadella - Microsoft Source
   https://news.microsoft.com/source/exec/satya-nadella/
   Satya Nadella is Chairman and Chief Executive Officer of Microsoft. Before being named CEO in February 2014, Nadella hel...

5. Asha Sharma - CEO XBOX - LinkedIn
   https://www.linkedin.com/in/aboutasha
   

'Top result: Satya Nadella - Wikipedia\nURL: https://en.wikipedia.org/wiki/Satya_Nadella\nSnippet: Satya Narayana Nadella (born 19 August 1967) is an Indian-American business executive. He is the chairman and chief executive officer (CEO) of Microsoft, ......'

graceful failure and retry logic

In [8]:
import time
def retry(func):
    def grace(*args):
        retry=3

        for attempt in range(retry):
            try:
                result=func(*args)
                return result
            except Exception as e:
                print(f"Retry {attempt +1}")
                time.sleep(1)
        
        return "Tool Failed"
    return grace

In [9]:
db_query_retry = retry(db_query)
web_search_retry=retry(web_search)

In [10]:
from langchain.tools import tool

In [11]:
@tool
def db_query_tool(employee_id: int)-> str:
    """ retrieve Employee information from database"""
    return db_query_retry(employee_id)

In [12]:
@tool
def web_search_tool(query:str)->str:
    """ search in the internet for inforamtion"""
    return web_search_retry(query)

In [13]:
print(web_search_tool.invoke("Sam Altman current company?"))

Web search results:
1. Sam Altman - Wikipedia
   https://en.wikipedia.org/wiki/Sam_Altman
   Samuel Harris Altman (born April 22, 1985) is an American entrepreneur and investor who has been the chief executive off...

2. Sam Altman - Forbes
   https://www.forbes.com/profile/sam-altman/
   Sam Altman is the CEO of OpenAI and a prolific venture investor. In 2005, he dropped out of Stanford to found social map...

3. Sam Altman - 2026 Portfolio & Founded Companies - Tracxn
   https://tracxn.com/d/people/sam-altman/__iX8odIfTKW_lGvpQK1yEz4sc2ld2VxU8EY90EfWiQlg
   Explore Sam Altman's startup portfolio, portfolio exits and founded companies across sectors and regions, along with lat...

4. Sam Altman's Portfolio: No OpenAI Equity, $1.7B Helion Stake
   https://www.startuphub.ai/ai-news/ai-figures/2026/figure-sam-altman-venture-portfolio-breakdown-2026-05-27
   Sam Altman's Portfolio: No OpenAI Equity, $1.7B Helion Stake May 2026 court filings show Sam Altman holds no direct Open...

5. Sam 

In [14]:
import boto3
import json
import os
from dotenv import load_dotenv


# AWS Credentials
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION")
load_dotenv("myenv.env")

# bedrock_runtime = boto3.client(
#     service_name="bedrock-runtime",
#     region_name=AWS_REGION,
#     aws_access_key_id=AWS_ACCESS_KEY_ID,
#     aws_secret_access_key=AWS_SECRET_ACCESS_KEY
# )

# MODEL_ID = "amazon.nova-micro-v1:0"

True

In [15]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model="amazon.nova-micro-v1:0",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

In [16]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[
        db_query_tool,
        web_search_tool
    ],
    system_prompt="""
    You are a ReAct agent.
    Use tools which are required.
    Think step-by-step before answering.
    """
)

In [17]:
def Agent(question):

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        }
    )

    return response["messages"][-1].content

In [18]:
answer = Agent("Who manages employee 01 and works as manager?")
print(answer)

<thinking> I have obtained the information that employee 01, whose name is "vinnu", is managed by "hari". Since "hari" is listed as the manager, it indicates that "hari" is the manager who works for employee 01. Therefore, the person who manages employee 01 and works as a manager is "hari". </thinking> 

The manager of employee 01 is "hari".


In [20]:
print(Agent("who manages employee 101 and work as manager?"))

<thinking> It appears that the database did not return any information about employee 101. Since there is no direct information available from the database, I should inform the user that there is no information about employee 101 in the database.</thinking>  

It seems that the database does not have information about employee 101. If you have additional context or another employee ID, I can try to retrieve that information. Otherwise, I am unable to provide details on who manages employee 101 and works as a manager based on the current tools available.


input topic filter

In [19]:
from pydantic import BaseModel, ValidationError
import re

ALLOWED_TOPICS={"employee","manager","organization","database"}

def topic_filter(q:str):
    if not any(t in q.lower() for t in ALLOWED_TOPICS):
        raise ValueError("Rejected: Topic not allowed.")
    return q

In [20]:
def pii_scrubber(text:str):
    text=re.sub(r'\b\d{3}-\d{2}-\d{4}\b','[SSN]',text)
    text=re.sub(r'[\w\.-]+@[\w\.-]+','[EMAIL]',text)
    text=re.sub(r'\b\d{10}\b','[PHONE]',text)
    return text

In [21]:
class AgentOutput(BaseModel):
    answer:str

def SafeAgent(question:str):
    question=topic_filter(question)
    question=pii_scrubber(question)
    raw=Agent(question)
    try:
        return AgentOutput(answer=raw).model_dump()
    except ValidationError as e:
        return {"error":str(e)}

In [22]:
tests=[
    "Who manages employee 01?",
    "What's the weather today?",
    "Employee email is abc@test.com"
]
for t in tests:
    try:
        print(t,'->',SafeAgent(t))
    except Exception as e:
        print(t,'->',e)

Who manages employee 01? -> {'answer': '<thinking> I have retrieved the manager information from the database for employee 01. The manager of employee 01 is "hari". I should now provide this information to the user. </thinking>\n\nThe manager of employee 01 is Hari.'}
What's the weather today? -> Rejected: Topic not allowed.
Employee email is abc@test.com -> {'answer': "<thinking> The User has provided an email address but has not specified what needs to be done with this information. To understand the context, I should ask the User for clarification on what they need assistance with regarding the email address. </thinking>\n\n\n\nHi there! It seems you've provided an email address. Could you please clarify what you need assistance with? Are you looking for more information related to this email or is there something specific you need help with?"}


In [23]:

from pydantic import BaseModel, ValidationError
import json

class EmployeeResponse(BaseModel):
    employee: str
    manager: str

def validate_json_output(output_text):
    try:
        data=json.loads(output_text)
        validated=EmployeeResponse(**data)
        print("✅ JSON schema validation passed")
        return validated.model_dump()
    except json.JSONDecodeError:
        raise ValueError("Agent output is not valid JSON")
    except ValidationError as e:
        raise ValueError(f"JSON schema validation failed:\n{e}")

# Example integration after agent.invoke(...)
# output = response["messages"][-1].content
# validated_output = validate_json_output(output)
# print(validated_output)


In [24]:
from pydantic import BaseModel, ValidationError
import json

class EmployeeResponse(BaseModel):
    employee: str
    manager: str


def validate_json_output(output):
    try:
        data = json.loads(output)
        validated = EmployeeResponse(**data)
        print("✅ JSON Schema Validation Passed")
        return validated.model_dump()

    except json.JSONDecodeError:
        print("❌ Output is not valid JSON")
        return None

    except ValidationError as e:
        print("❌ JSON Schema Validation Failed")
        print(e)
        return None

In [25]:
def SafeAgent(question):

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        }
    )

    output = response["messages"][-1].content

    return validate_json_output(output)

In [27]:
answer = SafeAgent("What is the name of employee 02?")
print(answer)

❌ Output is not valid JSON
None
